In [1]:
%cd drive/MyDrive/Thesis/Models/one_hot_NN

/content/drive/MyDrive/Thesis/Models/one_hot_NN


In [2]:
!ls

inference.ipynb		  MLP_predictions.npz  __pycache__  test_RNN.ipynb
Make_windowed_data.ipynb  model.py	       saved_model  training.ipynb


In [3]:
import numpy as np
import torch
import torch.nn as nn

from model import MLP_onehot_flat

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MLP_onehot_flat()
model.load_state_dict(torch.load('./saved_model', map_location = torch.device(device)))
model.eval()

MLP_onehot_flat(
  (l1): Linear(in_features=682, out_features=256, bias=True)
  (l2): Linear(in_features=256, out_features=256, bias=True)
  (l3): Linear(in_features=256, out_features=256, bias=True)
  (out): Linear(in_features=256, out_features=3, bias=True)
)

In [6]:
# test_set = np.load('../../Data/all_test_chains.npz')
test_set = np.load('../../Data/n62.npz')

In [7]:
window = 15
predictions = {}

for chain in test_set.files:
  seq = test_set[chain][0]
  SS  = test_set[chain][1]
  seqlen = len(test_set[chain][0])
  
  pred_seq = []

  for i in range (seqlen):
    if i > 14: 
        if i < seqlen - 16:
          model_input = seq[i-window:i+window+1] 
        else:
          model_input = np.append(seq[i-window:], np.full(i-seqlen+window+1, fill_value = 21, dtype='int8'))
    else:
      model_input = np.append(np.full(15-i, fill_value= 21, dtype = 'int8'), seq[0:i+window+1])

    model_input = torch.tensor(model_input, dtype=torch.int64)
    model_input = torch.nn.functional.one_hot(model_input, num_classes = 22)
    model_input = torch.flatten(model_input).to(dtype=torch.float)
    output      = model(model_input)

    _, pred     = torch.max(output, dim=0 )
    pred_seq.append(pred.item())

  predictions[chain] = (np.array(pred_seq))

  


In [ ]:
np.savez('../../Data/MLP_predictions.npz', **predictions)

In [8]:
np.savez('../../Data/MLP62.npz', **predictions)

In [ ]:
accuracies = []
for idx, chain in enumerate(test_set.files):
  assert (len(test_set[chain][1]) == len(predictions[idx]) )
  accuracies.append( (test_set[chain][1] == predictions[idx]).sum()/len(test_set[chain][1]) )



In [ ]:
np.mean(accuracies)

0.6627883010585137